# No-Show Prediction — Demo

Predicting whether a patient will miss a scheduled medical appointment, so clinic
staff can prioritise reminder and confirmation calls.

**Dataset:** Kaggle *Medical Appointment No Shows* — 110,527 appointments, 62,299
patients, public clinics in Vitória, Brazil, 2016.

Dataset by JoniHoppen (Aquarela Analytics), licensed [CC BY-NC-SA 4.0](https://creativecommons.org/licenses/by-nc-sa/4.0/) — attribution required, non-commercial use only.

This notebook runs top-to-bottom on a clean Google Colab runtime. It clones the
repo, downloads the dataset, trains the model, and demonstrates inference.

---

## Contents
1. [Setup](#setup)
2. [Data and cleaning](#data)
3. [What drives a no-show](#eda)
4. [Leakage safety](#leakage)
5. [Model results](#results)
6. [Inference demo](#inference)
7. [Limitations](#limitations)

<a id="setup"></a>
## 1. Setup

Clones the repo and installs dependencies. On Colab this takes about a minute.

In [ ]:
import os, sys, subprocess
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    if not Path("noshow-prediction-capstone").exists():
        subprocess.run(["git", "clone", "-q", "https://github.com/jamaliddins/noshow-prediction-capstone.git"], check=True)
    os.chdir("noshow-prediction-capstone")
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"],
        check=True,
    )
else:
    # Running locally from notebooks/ — step up to the repo root.
    if Path.cwd().name == "notebooks":
        os.chdir("..")

sys.path.insert(0, str(Path.cwd()))
print(f"Working directory: {Path.cwd()}")

### Dataset

The raw CSV is not committed — the repo tracks code only, and `.gitignore`
excludes `data/`. The cell below fetches it from a public mirror and verifies
the schema and row count match the documented dataset before continuing.

If the mirror is ever unavailable, download `KaggleV2-May-2016.csv` from
[Kaggle](https://www.kaggle.com/datasets/joniarroba/noshowappointments) and
place it at `data/KaggleV2-May-2016.csv`.

In [ ]:
import urllib.request
from pathlib import Path

DATA_PATH = Path("data/KaggleV2-May-2016.csv")
MIRROR = ("https://raw.githubusercontent.com/mroker242/no-show-appointments/"
          "master/noshowappointments-kagglev2-may-2016.csv")

DATA_PATH.parent.mkdir(parents=True, exist_ok=True)
if not DATA_PATH.exists():
    print("Downloading dataset...")
    urllib.request.urlretrieve(MIRROR, DATA_PATH)

# Fail loudly now rather than producing wrong numbers later.
EXPECTED_COLUMNS = [
    "PatientId", "AppointmentID", "Gender", "ScheduledDay", "AppointmentDay",
    "Age", "Neighbourhood", "Scholarship", "Hipertension", "Diabetes",
    "Alcoholism", "Handcap", "SMS_received", "No-show",
]
header = DATA_PATH.read_text(encoding="utf-8").split("\n", 1)[0].strip()
assert header.split(",") == EXPECTED_COLUMNS, f"Unexpected schema: {header}"

n_rows = sum(1 for _ in DATA_PATH.open(encoding="utf-8")) - 1
assert n_rows == 110_527, f"Expected 110,527 rows, got {n_rows:,}"

print(f"Dataset verified: {n_rows:,} rows, {len(EXPECTED_COLUMNS)} columns "
      f"({DATA_PATH.stat().st_size / 1e6:.1f} MB)")

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.dpi"] = 100

from src.preprocessing import load_raw, clean, build_features, split_by_patient, get_feature_columns
from src.config import TARGET
print("Imports OK")

<a id="data"></a>
## 2. Data and cleaning

One row is one scheduled appointment. The target is `No-show`: **1 = the patient
did not attend**.

Three problems are fixed here:
- `ScheduledDay` after `AppointmentDay` — logically impossible
- Impossible ages (negative, or above 110)
- Misspelled source columns (`Hipertension`, `Handcap`, `No-show`)

Note the date comparison is on **calendar dates**, not timestamps: an appointment
booked at 18:00 for that same day is legitimate, and comparing raw timestamps
would wrongly discard ~38,000 valid rows.

In [ ]:
raw = load_raw()
print(f"Raw:      {len(raw):,} appointments, {raw['patient_id'].nunique():,} patients")
print(f"No-show:  {raw[TARGET].mean():.2%}\n")

df = build_features(clean(raw))
print(f"\nAfter cleaning and feature engineering: {len(df):,} rows")
df[["age", "lead_time_days", "prior_noshow_rate", "sms_received", TARGET]].head()

<a id="eda"></a>
## 3. What drives a no-show

### 3.1 The classes are imbalanced

About 1 in 5 appointments is missed. This is why **accuracy is the wrong metric**:
always predicting "attended" scores ~80% accuracy while catching zero no-shows.
The primary metric is **F1 on the no-show class**.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
counts = df[TARGET].value_counts().sort_index()
ax.bar(["Attended", "No-show"], counts.values, color=["#4C8CBF", "#D1495B"], width=0.6)
for i, v in enumerate(counts.values):
    ax.text(i, v + 1500, f"{v:,}\n({v/counts.sum():.1%})", ha="center", fontweight="bold")
ax.set_ylabel("Appointments")
ax.set_title("Always guessing 'Attended' = 79.8% accuracy, 0 no-shows caught")
ax.set_ylim(0, counts.max() * 1.22)
plt.show()

### 3.2 Lead time is the strongest signal

The gap between booking and appointment matters more than any patient attribute.

In [ ]:
bins = [-0.5, 0.5, 1.5, 3.5, 7.5, 14.5, 30.5, np.inf]
labels = ["Same day", "1 day", "2-3", "4-7", "8-14", "15-30", "31+"]
rates = df.groupby(pd.cut(df["lead_time_days"], bins=bins, labels=labels),
                   observed=True)[TARGET].mean() * 100

fig, ax = plt.subplots(figsize=(9, 4.5))
ax.bar(rates.index.astype(str), rates.values,
       color=plt.cm.Reds(np.linspace(0.35, 0.85, len(rates))))
ax.axhline(df[TARGET].mean() * 100, ls="--", color="#333",
           label=f"Overall {df[TARGET].mean():.1%}")
for i, v in enumerate(rates.values):
    ax.text(i, v + 0.5, f"{v:.0f}%", ha="center", fontweight="bold")
ax.set_ylabel("No-show rate (%)"); ax.set_xlabel("Days waited")
ax.set_title("Same-day 5% → a month out 33%")
ax.legend(); plt.show()

### 3.3 The SMS paradox

Patients who received an SMS reminder miss **more** appointments — which looks
like the reminders backfire.

They do not. SMS is only sent when the wait is long, and long waits are what
drive no-shows. Conditioning on lead time reverses the effect: within every
lead-time band, SMS recipients attend *better*. This is a textbook confounder,
and it is the reason `SMS_received` was audited before being trusted as a feature.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

r = df.groupby("sms_received")[TARGET].mean() * 100
axes[0].bar(["No SMS", "SMS sent"], r.values, color=["#4C8CBF", "#D1495B"], width=0.55)
for i, v in enumerate(r.values):
    axes[0].text(i, v + 0.4, f"{v:.1f}%", ha="center", fontweight="bold")
axes[0].set_title("Raw: SMS looks harmful"); axes[0].set_ylabel("No-show rate (%)")

b = [-0.5, 0.5, 3.5, 7.5, 14.5, 30.5, np.inf]
l = ["Same day", "1-3", "4-7", "8-14", "15-30", "31+"]
g = (df.groupby([pd.cut(df["lead_time_days"], bins=b, labels=l), "sms_received"],
                observed=True)[TARGET].mean() * 100).unstack()
x = np.arange(len(g))
axes[1].bar(x - 0.2, g[0].values, 0.4, label="No SMS", color="#4C8CBF")
axes[1].bar(x + 0.2, g[1].values, 0.4, label="SMS sent", color="#D1495B")
axes[1].set_xticks(x); axes[1].set_xticklabels(g.index.astype(str), fontsize=9)
axes[1].set_title("Conditioned on lead time: the effect reverses")
axes[1].set_xlabel("Lead time (days)"); axes[1].legend()
plt.tight_layout(); plt.show()

<a id="leakage"></a>
## 4. Leakage safety

Two forms of leakage would inflate the score on this dataset. Both are prevented,
and both are checked here rather than asserted.

**4.1 Patient-grouped split.** 62,299 patients generate 110,527 appointments, so
a random row split would put the same patient in both train and test — the model
could memorise individuals. The split is grouped on `PatientId`.

**4.2 History features use only the past.** `prior_noshow_rate` is computed from
appointments scheduled *strictly before* the row being predicted. Using the full
history would leak the outcome into its own features.

In [ ]:
train, val, test = split_by_patient(df)

print("Split sizes")
for name, part in [("train", train), ("val", val), ("test", test)]:
    print(f"  {name:<6}{len(part):>7,} rows  {part['patient_id'].nunique():>6,} patients"
          f"  no-show {part[TARGET].mean():.2%}")

tr, va, te = (set(p["patient_id"]) for p in (train, val, test))
print(f"\nPatient overlap — train/val: {len(tr & va)}, train/test: {len(tr & te)},"
      f" val/test: {len(va & te)}")
assert not (tr & va) and not (tr & te) and not (va & te)
print("PASS: no patient appears in more than one split")

In [ ]:
# Proof that a row's own outcome cannot reach its own features: flip the label
# on one appointment and confirm that row's history features do not move.
from src.preprocessing import add_patient_history_features

repeat_patient = df[df["prior_appointments"] > 2]["patient_id"].iloc[0]
subset = df[df["patient_id"] == repeat_patient].copy()
target_row = subset["appointment_id"].iloc[-1]

before = add_patient_history_features(subset).set_index("appointment_id")
flipped = subset.copy()
flipped.loc[flipped["appointment_id"] == target_row, TARGET] ^= 1
after = add_patient_history_features(flipped).set_index("appointment_id")

cols = ["prior_appointments", "prior_noshows", "prior_noshow_rate"]
print(f"Patient {repeat_patient}, appointment {target_row}")
print(pd.DataFrame({
    "original label": before.loc[target_row, cols],
    "flipped label":  after.loc[target_row, cols],
}))
assert (before.loc[target_row, cols] == after.loc[target_row, cols]).all()
print("\nPASS: features are identical — the row's own outcome does not leak")

<a id="results"></a>
## 5. Model results

Four models, each scored on validation; the winner is scored **once** on the
held-out test set. Every later model must beat Logistic Regression.

Training takes roughly two minutes on Colab.

In [ ]:
from src.train import train_all

results = train_all(log_to_mlflow=False)

### 5.1 Reading the numbers

**F1 = 0.46 is a modest score, and that is the honest ceiling here.** No feature
correlates above 0.28 with the target; human behaviour is only partly predictable
from administrative records. What matters operationally is the separation between
tiers, shown below.

Two checks worth noting:
- The **val → test gap is near zero**, so the patient-grouped split held.
- Probabilities are **calibrated**: class weighting inflated them to a mean of
  0.43 against a true rate of 0.20, so isotonic regression was fitted on
  validation. A displayed "40%" now means roughly a 40% chance.

In [ ]:
best = results["_best"]
m = best["test"]

print(f"Selected model: {best['name']} (calibrated), threshold {best['threshold']:.3f}\n")
print(f"  F1        {m['f1']:.4f}   <- primary metric")
print(f"  Precision {m['precision']:.4f}")
print(f"  Recall    {m['recall']:.4f}")
print(f"  PR-AUC    {m['pr_auc']:.4f}")
print(f"  ROC-AUC   {m['roc_auc']:.4f}")
print(f"\n  Of {m['tp'] + m['fn']:,} real no-shows, caught {m['tp']:,} ({m['recall']:.1%}).")
print(f"  Flagged {m['tp'] + m['fp']:,} appointments; {m['precision']:.1%} were real no-shows.")

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay, precision_recall_curve

y_test = results["_data"]["y_test"]
y_prob = results["_data"]["test_prob"]
t = best["threshold"]

fig, axes = plt.subplots(1, 2, figsize=(13, 4.8))
ConfusionMatrixDisplay.from_predictions(
    y_test, (y_prob >= t).astype(int), display_labels=["Attended", "No-show"],
    cmap="Blues", colorbar=False, values_format=",", ax=axes[0],
)
axes[0].set_title(f"Confusion matrix (t = {t:.3f})"); axes[0].grid(False)

prec, rec, _ = precision_recall_curve(y_test, y_prob)
axes[1].plot(rec, prec, color="#D1495B", lw=2.5, label=f"PR-AUC = {m['pr_auc']:.3f}")
axes[1].axhline(y_test.mean(), ls="--", color="grey", label=f"Random = {y_test.mean():.3f}")
axes[1].plot(m["recall"], m["precision"], "o", ms=11, color="#2A9D8F", label="Operating point")
axes[1].set_xlabel("Recall"); axes[1].set_ylabel("Precision")
axes[1].set_title("Precision-Recall curve"); axes[1].legend(fontsize=9)
plt.tight_layout(); plt.show()

<a id="inference"></a>
## 6. Inference demo

This is what clinic staff would use: a probability, a risk tier, and a suggested
action.

In [ ]:
from src.predict import predict_one, predict_batch, InvalidAppointmentError

appointment = {
    "scheduled_day": "2016-05-02T09:00:00",
    "appointment_day": "2016-05-30",   # 28-day wait
    "age": 22,
    "gender": "F",
    "neighbourhood": "JARDIM CAMBURI",
    "scholarship": 1,
    "sms_received": 1,
}

result = predict_one(appointment)
for k, v in result.items():
    print(f"  {k:<22} {v}")

In [ ]:
# Risk gauge for a single appointment.
prob = result["no_show_probability"]
tier = result["risk_tier"]
color = {"Low": "#52B788", "Medium": "#F4A261", "High": "#D1495B"}[tier]

fig, ax = plt.subplots(figsize=(9, 1.9))
ax.barh([0], [100], color="#EEEEEE", height=0.5)
ax.barh([0], [prob * 100], color=color, height=0.5)
ax.axvline(result["threshold_used"] * 100, color="#333", ls="--", lw=1.5)
ax.text(result["threshold_used"] * 100 + 1, 0.34, "decision threshold",
        fontsize=8, color="#333")
ax.text(prob * 100 / 2, 0, f"{prob*100:.1f}%", ha="center", va="center",
        color="white", fontweight="bold", fontsize=15)
ax.set_xlim(0, 100); ax.set_ylim(-0.5, 0.5)
ax.set_yticks([]); ax.set_xlabel("No-show probability (%)")
ax.set_title(f"{tier} risk — {result['recommendation']}", fontweight="bold")
ax.grid(False)
plt.tight_layout(); plt.show()

### 6.1 A batch of tomorrow's appointments

Sorted highest-risk first, which is the order staff would work through.

In [ ]:
clinic_day = [
    {"scheduled_day": "2016-05-02", "appointment_day": "2016-05-30", "age": 22,
     "gender": "F", "neighbourhood": "JARDIM CAMBURI", "scholarship": 1},
    {"scheduled_day": "2016-05-30T08:00", "appointment_day": "2016-05-30", "age": 65,
     "gender": "M", "neighbourhood": "CENTRO", "hypertension": 1},
    {"scheduled_day": "2016-05-10", "appointment_day": "2016-05-24", "age": 15,
     "gender": "M", "neighbourhood": "SANTOS DUMONT"},
    {"scheduled_day": "2016-05-20", "appointment_day": "2016-05-23", "age": 45,
     "gender": "F", "neighbourhood": "MARIA ORTIZ"},
    {"scheduled_day": "2016-04-15", "appointment_day": "2016-05-30", "age": 31,
     "gender": "F", "neighbourhood": "ITARARE", "scholarship": 1, "sms_received": 1},
    {"scheduled_day": "2016-05-29", "appointment_day": "2016-05-30", "age": 78,
     "gender": "F", "neighbourhood": "MARUIPE", "diabetes": 1, "hypertension": 1},
]

batch = predict_batch(clinic_day)
batch

In [ ]:
colors = {"Low": "#52B788", "Medium": "#F4A261", "High": "#D1495B"}
fig, ax = plt.subplots(figsize=(10, 4))
y = np.arange(len(batch))[::-1]
ax.barh(y, batch["no_show_percentage"], color=[colors[t] for t in batch["risk_tier"]])
ax.set_yticks(y)
ax.set_yticklabels([f"{r.age}y {r.gender}, {r.lead_time_days}d wait"
                    for r in batch.itertuples()], fontsize=9)
for yi, (pct, tier) in enumerate(zip(batch["no_show_percentage"], batch["risk_tier"])):
    ax.text(pct + 1, y[yi], f"{pct:.1f}%  {tier}", va="center", fontsize=9,
            fontweight="bold")
ax.set_xlabel("No-show probability (%)")
ax.set_title("Call list, highest risk first", fontweight="bold")
ax.set_xlim(0, max(batch["no_show_percentage"]) * 1.35)
plt.tight_layout(); plt.show()

### 6.2 Invalid input is rejected, not guessed at

The brief requires incomplete or invalid input to be handled. Bad records raise a
clear error rather than producing a silent, meaningless prediction.

In [ ]:
bad_records = [
    ({"age": 30, "gender": "F"}, "missing required fields"),
    ({"scheduled_day": "2016-06-01", "appointment_day": "2016-05-01", "age": 30,
      "gender": "F", "neighbourhood": "CENTRO"}, "appointment before booking"),
    ({"scheduled_day": "2016-05-01", "appointment_day": "2016-05-10", "age": 999,
      "gender": "F", "neighbourhood": "CENTRO"}, "impossible age"),
    ({"scheduled_day": "2016-05-01", "appointment_day": "2016-05-10", "age": 30,
      "gender": "X", "neighbourhood": "CENTRO"}, "unknown gender"),
]

for record, description in bad_records:
    try:
        predict_one(record)
        print(f"  NOT REJECTED ({description}) — this would be a bug")
    except InvalidAppointmentError as exc:
        print(f"  rejected [{description}]: {exc}")

### 6.3 What the tiers mean operationally

This is the result that matters to the clinic. The model does not need to be
highly accurate to be useful — it needs to *rank* well enough that staff effort
lands where it pays off.

In [ ]:
import json
meta = json.loads(open("models/model_metadata.json").read())
tiers = meta["risk_tiers"]

labels = np.where(y_prob < tiers["low_max"], "Low",
                  np.where(y_prob < tiers["medium_max"], "Medium", "High"))

rows = []
for tier in ["Low", "Medium", "High"]:
    mask = labels == tier
    rows.append({
        "Tier": tier,
        "Appointments": f"{mask.sum():,}",
        "Share": f"{mask.mean():.1%}",
        "Actual no-show rate": f"{y_test[mask].mean():.1%}",
    })
summary = pd.DataFrame(rows)

lift = y_test[labels == "High"].mean() / y_test[labels == "Low"].mean()
print(summary.to_string(index=False))
print(f"\nHigh-risk patients miss {lift:.1f}x more often than Low-risk.")
print(f"Calling the top {(labels == 'High').mean():.0%} of appointments reaches "
      f"{y_test[labels == 'High'].sum() / y_test.sum():.0%} of all no-shows.")

<a id="limitations"></a>
## 7. Limitations and next steps

**Limitations**

1. **One city, one year.** Vitória, Brazil, 2016. Nothing here demonstrates that
   the model transfers to another health system, and it should be retrained
   before use elsewhere.
2. **Modest discrimination.** F1 ≈ 0.46, ROC-AUC ≈ 0.75. Useful for ranking, not
   for confident individual judgements.
3. **Precision is ~34%.** Roughly two in three flagged patients would have
   attended anyway. Acceptable when the action is a phone call; not acceptable
   for anything punitive.
4. **Fairness.** `Neighbourhood` and `Scholarship` are socioeconomic proxies, and
   no-show rates do vary by neighbourhood. The model must not be used to
   deprioritise, penalise, or deny care to anyone.
5. **SMS timing is not fully documented.** The dataset does not record when the
   SMS was sent. The confounding analysis in section 3.3 is consistent with it
   being pre-appointment, but this could not be verified from the data alone.
6. **No appointment type or specialty**, which likely matter and are absent here.

**Appropriate use:** a prioritisation aid for reminder calls, with a human in the
loop. Not a clinical decision-maker, and not grounds for cancelling or
deprioritising an appointment.

**Next steps**

- Validate on data from another clinic or year before any deployment
- Add specialty, appointment type, and distance-to-clinic if available
- Run a controlled trial: does calling the High tier actually reduce no-shows?
- Monitor per-neighbourhood performance for disparities in production

---

## Summary

| | |
|---|---|
| **Task** | Binary classification — will this patient miss their appointment? |
| **Data** | 110,527 appointments, 62,299 patients, cleaned to 110,516 rows |
| **Split** | 70/15/15 grouped by patient, verified zero overlap |
| **Models** | Majority baseline → Logistic Regression → Random Forest → XGBoost |
| **Primary metric** | F1 on the no-show class |
| **Test result** | F1 0.453, PR-AUC 0.374, ROC-AUC 0.743, recall 77.6% |
| **Operational value** | High tier = 10% of appointments, 4.5× the no-show rate |
| **Output** | Calibrated probability + Low/Medium/High tier + suggested action |